# 🧪 Praktikum Alternatif Bab 07: Evaluasi Mendalam Deteksi Serangan Siber (Cyber Intrusion Detection)
### Universitas Indraprasta PGRI (Unindra) - Laboratorium Informatika
**Mata Kuliah**: Machine Learning  
**Modul Referensi**: Bab 07  
**Fokus Topik**: Confusion Matrix, ROC-AUC, PR-Curve & Cost-Matrix (Studi Kasus: Keamanan Jaringan)  

---

## 📌 Perbedaan dengan Modul Utama:
> Berbeda dari kasus Heart Disease modul. Pada deteksi serangan siber, False Negative (serangan lolos tidak terdeteksi) berakibat fatal bagi keamanan server. Kita membedah pemilihan threshold optimal melalui ROC & Precision-Recall curve.

---

## 🎯 Tujuan Praktikum:
1. Memahami penerapan algoritma pada studi kasus riil yang kontekstual dan mutakhir.
2. Melatih pembersihan, eksplorasi, transformasi, hingga evaluasi metrik kinerja model.
3. Menumbuhkan daya nalar komputasi mahasiswa di luar dataset standar modul kuliah.


## 1. Persiapan Environment & Import Pustaka

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc, precision_recall_curve


## 2. Inisialisasi & Karakterisasi Dataset Alternatif

In [ ]:
# Dataset Simulasi Log Serangan Siber
np.random.seed(88)
n_packets = 1000

packet_rate = np.random.exponential(scale=50, size=n_packets)
error_rate = np.random.uniform(0.0, 0.5, size=n_packets)
failed_logins = np.random.poisson(lam=0.8, size=n_packets)
duration_sec = np.random.exponential(scale=10, size=n_packets)

# Serangan = DDoS / Brute Force
is_attack = ((packet_rate > 150) | (failed_logins >= 4) | ((error_rate > 0.3) & (packet_rate > 70))).astype(int)

df_cyber = pd.DataFrame({
    'packet_rate_per_sec': np.round(packet_rate, 1),
    'byte_error_rate': np.round(error_rate, 3),
    'failed_login_attempts': failed_logins,
    'session_duration_sec': np.round(duration_sec, 1),
    'is_intrusion': is_attack
})

df_cyber.to_csv('../../data/raw/alternatif/cyber_intrusion_logs.csv', index=False)
print("Persentase Serangan:", df_cyber['is_intrusion'].mean() * 100, "%")
df_cyber.head()


## 3. Implementasi Algoritma & Analisis Kinerja Model

In [ ]:
X = df_cyber.drop('is_intrusion', axis=1)
y = df_cyber['is_intrusion']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_proba = model.predict_proba(X_test)[:, 1]
fpr, tpr, _ = roc_curve(y_test, y_proba)
roc_auc = auc(fpr, tpr)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Confusion Matrix Default Threshold 0.5
cm = confusion_matrix(y_test, (y_proba > 0.5).astype(int))
ConfusionMatrixDisplay(cm, display_labels=['Normal', 'Intrusion']).plot(ax=axes[0], cmap='Blues')
axes[0].set_title('Confusion Matrix (Threshold = 0.5)')

# 2. ROC Curve
axes[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.3f})')
axes[1].plot([0, 1], [0, 1], color='navy', linestyle='--')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate (Recall)')
axes[1].set_title('Receiver Operating Characteristic (ROC)')
axes[1].legend(loc="lower right")
axes[1].grid(True, alpha=0.5)

plt.tight_layout()
plt.show()


## 4. Tugas Mandiri & Pertanyaan Diskusi Laboratorium:
1. **Analisis Sensitivitas**: Bagaimana perubahan hyperparameter mempengaruhi metrik evaluasi pada dataset ini?
2. **Kesesuaian Algoritma**: Mengapa algoritma yang digunakan cocok untuk karakteristik data di studi kasus ini?
3. **Pertimbangan Bisnis/Etika**: Jika model ini diimplementasikan di dunia nyata, apa risiko kesalahan prediksi (False Positive vs False Negative) yang paling berbahaya?
